# Building a Transformer from Scratch
### Part 2: Self-Attention & Multi-Head Attention

In Part 1 we built a Bigram model — it only looked at the previous character to predict the next one. That's too limited.

In this notebook we add **self-attention**, which lets every token look at all previous tokens and decide which ones matter most.

**What you will learn:**
- what Q, K, V are and why we need all three
- how causal masking prevents tokens from seeing the future
- why we scale attention scores by `1/√head_size`
- how multi-head attention runs several attention heads in parallel

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

## 1. Data

In [ ]:
with open('santi.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    chars = sorted(set(text))
    vocab_size = len(chars)

print(f'vocab size: {vocab_size}')

In [ ]:
stoi = {s: i for i, s in enumerate(chars)}
itos = {i: s for s, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data  # fixed: was 'train_data'
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

## 2. Why Scaling Matters

Before building attention, here's a key insight — why we scale attention scores by `1/√head_size`.

When attention scores are large, softmax becomes very peaked (one token gets all the weight). Scaling keeps the scores small so softmax stays spread out and the model can attend to multiple tokens at once.

In [ ]:
torch.manual_seed(1145)

a = torch.randn(4, 4)        # attention scores, mean=0 std=1
a_scaled = a * 4 ** -0.5     # scale down by 1/√head_size

print('before scaling:')
print(F.softmax(a, dim=-1))        # peaked — one token dominates
print('after scaling:')
print(F.softmax(a_scaled, dim=-1)) # more uniform — attends to multiple tokens

## 3. Self-Attention Head

Each token produces three vectors:
- **Q (query)** — "what am I looking for?"
- **K (key)** — "what do I contain?"
- **V (value)** — "what do I actually communicate?"

Attention score = `Q · Kᵀ` — how well each query matches each key.

The **causal mask** sets future positions to `-inf` before softmax, so they become 0 after softmax. This means token at position 3 can only attend to positions 0, 1, 2, 3 — never the future.

In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.W_q = nn.Linear(n_embd, head_size, bias=False)  # query projection
        self.W_k = nn.Linear(n_embd, head_size, bias=False)  # key projection
        self.W_v = nn.Linear(n_embd, head_size, bias=False)  # value projection

    def forward(self, x):
        B, T, C = x.shape  # C = n_embd

        q = self.W_q(x)  # (B, T, head_size)
        k = self.W_k(x)
        v = self.W_v(x)

        weight = q @ k.transpose(-2, -1)          # (B, T, T) — attention scores
        weight = weight * (q.shape[-1]) ** (-0.5) # scale to prevent softmax collapse
        tril = torch.tril(torch.ones(T, T))
        weight = weight.masked_fill(tril == 0, float('-inf'))  # causal mask
        weight = F.softmax(weight, dim=-1)         # normalize
        out = weight @ v                           # (B, T, head_size)

        return out

## 4. Multi-Head Attention

Instead of one attention head, we run several in parallel — each head can focus on different relationships between tokens.

The outputs are concatenated and projected back to `n_embd`:
```
n_head heads × head_size each → concat → (n_head * head_size) → proj → n_embd
```

Note: `head_size = n_embd // n_head` so the total stays at `n_embd`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, head_size, n_head, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embd, head_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_head * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

## 5. Model

Same structure as Part 1 but with multi-head attention replacing the simple bigram lookup.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_head, n_embd):
        super().__init__()
        head_size = n_embd // n_head  # fixed: was hardcoded to 16
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.multiheadattention = MultiHeadAttention(head_size, n_head, n_embd)
        self.project_out = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)      # (B, T, n_embd)
        logits = self.multiheadattention(logits)       # (B, T, n_embd)
        logits = self.project_out(logits)              # (B, T, vocab_size)
        loss = None

        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]              # (B, vocab_size)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 6. Training

Hyperparameters: `n_embd=32, n_head=4` → `head_size = 32//4 = 8` per head.

In [ ]:
n_embd = 32   # fixed: was 8 with n_head=16 which is impossible (8/16 < 1)
n_head = 4    # head_size = 32//4 = 8 per head
eval_iters = 200
eval_interval = 500

model = BigramLanguageModel(vocab_size, n_head=n_head, n_embd=n_embd)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print(f'model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
for steps in range(5000):
    if steps % eval_interval == 0:
        losses = estimate_loss(model, eval_iters)
        print(f"step {steps}: train {losses['train']:.4f}, val {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from scratch
idx = torch.zeros((1, 1), dtype=torch.long)  # fixed: was idx = xb
print(decode(model.generate(idx, max_new_tokens=200)[0].tolist()))

## What's Next?

This model has attention but is still missing a lot:
- no positional embeddings — tokens don't know their position
- no LayerNorm or residual connections — training is unstable at depth
- no feedforward layers — limited capacity

**Part 3** puts all of this together into a full transformer, adds RoPE for position, and trains on 三体 with W&B experiment tracking.